# Бустинг: от идеи к практике

В этом ноутбуке мы разберёмся с семейством методов **бустинга** — мощных ансамблей, которые последовательно улучшают модель.

**Цели ноутбука:**
- Понять разницу между бэггингом (случайный лес) и бустингом.
- Разобрать, как устроен градиентный бустинг.
- Посмотреть на `GradientBoostingClassifier` и `AdaBoostClassifier` из `scikit-learn` на синтетических данных.

---

**Содержание:** Бэггинг vs бустинг → Градиентный бустинг → AdaBoost → Данные → Сравнение моделей.


## Бэггинг vs бустинг

**Бэггинг (bagging)**, как в случайном лесе:
- обучаем **много независимых** моделей (деревьев) параллельно;
- каждое дерево видит случайную подвыборку объектов и/или признаков;
- итог — усреднение/голосование.

Цель бэггинга — **уменьшить дисперсию** (variance), сгладить переобучение одиночной модели.

**Бустинг (boosting)**:
- обучаем модели **последовательно**;
- каждая новая модель старается **исправить ошибки** предыдущих;
- итоговый прогноз — взвешенная сумма/комбинация всех базовых моделей.

Цель бустинга — поэтапно уменьшать **смещение** (bias), получая всё более точную модель.


## Идея градиентного бустинга

Упрощённо градиентный бустинг для регрессии можно представить так:
1. Стартуем с простой модели $f_0(x)$ (например, константа — среднее по `y`).
2. Считаем **остатки**: $r_i = y_i - f_0(x_i)$.
3. Обучаем слабую модель (например, маленькое дерево) предсказывать эти остатки.
4. Обновляем модель: $f_1(x) = f_0(x) + \eta \cdot h_1(x)$, где $h_1$ — слабая модель, $\eta$ — шаг обучения (`learning_rate`).
5. Повторяем шаги 2–4 много раз.

Для классификации всё немного сложнее, но идея та же: на каждом шаге мы двигаемся в направлении, которое уменьшает функцию потерь (отсюда слово **"градиентный"**).


In [1]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=400, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_train.shape, X_test.shape


((280, 2), (120, 2))

## `GradientBoostingClassifier` на примере

Используем реализацию градиентного бустинга из `sklearn`. Поиграем с параметрами:
- `n_estimators` — число базовых моделей (обычно небольших деревьев);
- `learning_rate` — насколько сильно каждый новый шаг влияет на итоговую модель;
- `max_depth` базовых деревьев (через `max_depth` параметра `DecisionTreeClassifier` внутри бустинга).


In [2]:
from sklearn.ensemble import GradientBoostingClassifier

def evaluate_gb(n_estimators: int, learning_rate: float, max_depth: int = 3, random_state: int = 42):
    clf = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=random_state,
    )
    clf.fit(X_train, y_train)
    y_pred_train = clf.predict(X_train)
    y_pred_test = clf.predict(X_test)
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    print(
        f"GB: n_estimators={n_estimators:3d}, lr={learning_rate:.2f}, max_depth={max_depth} | "
        f"train={train_acc:.3f}, test={test_acc:.3f}"
    )
    return clf

for n_estimators in [10, 50, 100]:
    _ = evaluate_gb(n_estimators=n_estimators, learning_rate=0.1, max_depth=3)


GB: n_estimators= 10, lr=0.10, max_depth=3 | train=0.893, test=0.875
GB: n_estimators= 50, lr=0.10, max_depth=3 | train=0.954, test=0.917
GB: n_estimators=100, lr=0.10, max_depth=3 | train=0.989, test=0.908


## `AdaBoostClassifier`: классический бустинг

`AdaBoost` — один из первых и концептуально простых вариантов бустинга. Его удобно понимать как **последовательное взвешенное голосование слабых моделей**.

### Схема AdaBoost

Рассмотрим бинарную классификацию, где метки классов записаны как $y_i \in \{-1, +1\}$.

1. **Назначаем начальные веса объектам.**  Всем объектам даём одинаковый вес: каждый пример "одинаково важен".
2. **Обучаем слабый классификатор $h_1(x)$** на тех же данных, но учитывая веса: ошибка на более тяжёлых объектах дороже.
3. **Считаем взвешенную ошибку** этой модели и назначаем ей вес $\alpha_1$ в итоговом голосовании (чем меньше ошибка, тем больше $\alpha_1$).
4. **Обновляем веса объектов:**
   - если объект классифицирован **правильно**, его вес уменьшается;
   - если **ошибочно** — вес увеличивается.  Так следующий классификатор сильнее сфокусируется на трудных примерах.
5. Повторяем шаги 2–4 много раз, получая $h_2, h_3, \dots, h_M$.
6. **Финальное решение** строим как знак взвешенной суммы всех слабых моделей:
   $$
   F(x) = \operatorname{sign}\left(\sum_{m=1}^M \alpha_m h_m(x)\right).
   $$

Таким образом, каждый новый слабый классификатор пытается исправить те примеры, где совокупность предыдущих моделей ошибалась, а итоговая модель — это их взвешенное голосование.

### Как именно обновляются веса объектов (идея без вывода)

Пусть на шаге $m$ у нас есть веса $w_i^{(m)}$ и новый слабый классификатор $h_m(x)$ с весом $\alpha_m$.

- Если объект классифицирован **правильно** (то есть $h_m(x_i) = y_i$), его новый вес уменьшается:
  $$
  w_i^{(m+1)} \propto w_i^{(m)} e^{-\alpha_m}.
  $$
- Если объект классифицирован **ошибочно** (то есть $h_m(x_i) \ne y_i$), его новый вес увеличивается:
  $$
  w_i^{(m+1)} \propto w_i^{(m)} e^{+\alpha_m}.
  $$

После этого все новые веса нормируют так, чтобы $\sum_i w_i^{(m+1)} = 1$. Чем лучше слабый классификатор (меньше его ошибка и больше $\alpha_m$), тем сильнее перераспределение весов между правильными и ошибочными объектами.

В `sklearn` всё это уже реализовано в `AdaBoostClassifier`, поэтому в коде ниже мы используем готовую реализацию и играем параметрами `n_estimators` и `learning_rate`. 


In [3]:
from sklearn.ensemble import AdaBoostClassifier

def evaluate_ada(n_estimators: int, learning_rate: float, random_state: int = 42):
    clf = AdaBoostClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=random_state,
    )
    clf.fit(X_train, y_train)
    y_pred_train = clf.predict(X_train)
    y_pred_test = clf.predict(X_test)
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    print(
        f"AdaBoost: n_estimators={n_estimators:3d}, lr={learning_rate:.2f} | "
        f"train={train_acc:.3f}, test={test_acc:.3f}"
    )
    return clf

for n_estimators in [10, 50, 100]:
    _ = evaluate_ada(n_estimators=n_estimators, learning_rate=0.5)


AdaBoost: n_estimators= 10, lr=0.50 | train=0.868, test=0.842
AdaBoost: n_estimators= 50, lr=0.50 | train=0.911, test=0.908
AdaBoost: n_estimators=100, lr=0.50 | train=0.914, test=0.917


## Когда использовать бустинг, а когда лес

- **Случайный лес**:
  - хорошо работает "из коробки" с минимальной настройкой;
  - устойчив к шуму и выбросам;
  - проще интерпретировать важности признаков.

- **Бустинг (GBM/XGBoost/LightGBM/CatBoost)**:
  - часто даёт **лучшее качество**, особенно на табличных данных;
  - более чувствителен к настройке гиперпараметров (learning_rate, глубина деревьев, число итераций);
  - может переобучаться при слишком большом числе итераций и большом шаге.

Типичный рабочий сценарий: начать со случайного леса как с базовой модели, затем переходить к бустингу для достижения максимального качества.


## Выводы

- Бустинг обучает модели последовательно, каждая следующая исправляет ошибки предыдущих; в итоге — взвешенная комбинация (в отличие от параллельного бэггинга в случайном лесе).

- На практике для табличных данных часто используют бустинг (XGBoost, LightGBM, CatBoost) после базовой оценки случайным лесом.